In [1]:
from pathlib import Path
import os
import re
import random
import warnings

import numpy as np
import pandas as pd
import scanpy as sc
from scipy.sparse import issparse
import matplotlib.pyplot as plt

from llm_sc_curator import LLMscCurator
from llm_sc_curator.backends import BaseLLMBackend
from llm_sc_curator.noise_lists import NOISE_PATTERNS, NOISE_LISTS

from benchmarks.gt_mappings import (
    get_cd8_ground_truth,
    get_cd4_ground_truth,
    get_msc_ground_truth,
    get_bcell_ground_truth,
)


In [2]:
warnings.filterwarnings("ignore")

RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
os.environ["PYTHONHASHSEED"] = str(RANDOM_SEED)

BASE = Path("/work")
INPUT_DIR = BASE / "paper" / "gb_resubmission" / "input"
OUTPUT_DIR = BASE / "paper" / "gb_resubmission" / "output" / "FigS3_cross_dataset"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

TOPN_GRID = [10, 20, 30, 50]
N_TOP = 50

print("INPUT_DIR:", INPUT_DIR)
print("OUTPUT_DIR:", OUTPUT_DIR)

INPUT_DIR: /work/paper/gb_resubmission/input
OUTPUT_DIR: /work/paper/gb_resubmission/output/FigS3_cross_dataset


In [3]:
# ---------------------------------------------------------------------
# Dummy backend: curate_features / set_global_context only; no LLM calls
# ---------------------------------------------------------------------
class DummyBackend(BaseLLMBackend):
    def generate(self, prompt: str, json_mode: bool = False) -> str:
        return '{"cell_type":"Dummy","confidence":"Low","reasoning":"Not used in backend-free evaluation"}'


# ---------------------------------------------------------------------
# Dataset configs
# ---------------------------------------------------------------------
DATASET_CONFIGS = [
    {
        "dataset": "CD8",
        "path": INPUT_DIR / "cd8_benchmark_data.h5ad",
        "cluster_col": "meta.cluster",
        "gt_fn": get_cd8_ground_truth,
        "exclude_gt": {"CD8_Other", "Other", "Unknown"},
    },
    {
        "dataset": "CD4",
        "path": INPUT_DIR / "cd4_benchmark_data.h5ad",
        "cluster_col": "meta.cluster",
        "gt_fn": get_cd4_ground_truth,
        "exclude_gt": {"CD4_Other", "Other", "Unknown"},
    },
    {
        "dataset": "MSC",
        "path": INPUT_DIR / "brca_msc_benchmark_data.h5ad",
        "cluster_col": "meta.cluster",
        "gt_fn": get_msc_ground_truth,
        "exclude_gt": {"Fibro_Other", "Other", "Unknown"},
    },
    {
        "dataset": "MOUSE_B",
        "path": INPUT_DIR / "mouse_b_benchmark_data.h5ad",
        "cluster_col": "meta.cluster",
        "gt_fn": get_bcell_ground_truth,
        "exclude_gt": {"B_Other", "Other", "Unknown"},
    },
]

In [4]:
# ---------------------------------------------------------------------
# Canonical marker DBs for backend-free recall
# Keep these compact and interpretable.
# ---------------------------------------------------------------------
CD8_MARKER_DB = {
    "CD8_Naive": {
        "IL7R", "CCR7", "LEF1", "TCF7", "SELL", "MAL", "LTB", "KLF2"
    },
    "CD8_EffectorMemory": {
        "GZMK", "LTB", "AQP3", "IL7R", "CXCR4", "ANXA1", "ZFP36L2", "DUSP2"
    },
    "CD8_Effector": {
        "CCL5", "NKG7", "PRF1", "GZMB", "CX3CR1", "KLRG1", "FGFBP2", "GNLY"
    },
    "CD8_Exhausted": {
        "CXCL13", "CTLA4", "TIGIT", "HAVCR2", "PDCD1", "ENTPD1", "TOX", "TNFRSF9"
    },
    "CD8_ISG": {
        "IFIT1", "ISG15", "MX1", "STAT1", "OAS1", "IFI6"
    },
    "CD8_MAIT": {
        "SLC4A10", "KLRB1", "CXCR6"
    },
    "CD8_Cycling": {
        "MKI67", "TOP2A", "CDK1", "BIRC5", "PCNA", "TYMS"
    },
    "CD8_NK_Killer": {
        "NKG7", "GNLY", "PRF1", "GZMB", "FGFBP2"
    },
}

CD4_MARKER_DB = {
    "CD4_Treg": {"FOXP3", "IL2RA", "CTLA4", "TIGIT", "IKZF2", "BATF"},
    "CD4_Tfh": {"CXCL13", "CXCR5", "PDCD1", "ICOS", "TOX", "SH2D1A"},
    "CD4_Th17": {"KLRB1", "CCR6", "IL7R", "RORC", "IL23R", "RORA"},
    "CD4_ISG": {"IFIT1", "ISG15", "MX1", "STAT1", "IFI6", "OAS1"},
    "CD4_Cycling": {"MKI67", "TOP2A", "CDK1", "BIRC5", "PCNA", "TYMS"},
    "CD4_Exhausted": {"PDCD1", "TOX", "CTLA4", "TIGIT", "LAG3", "HAVCR2"},
    "CD4_Tn.Naive": {"IL7R", "CCR7", "LEF1", "TCF7", "SELL", "LTB", "KLF2", "MAL"},
    "CD4_Tem.EffMem": {"IL7R", "AQP3", "LTB", "ANXA1", "GZMK", "MAL"},
    "CD4_Temra.EffMem": {"CCL5", "PRF1", "NKG7", "FGFBP2", "GZMB", "CX3CR1"},
    "CD4_Tm.EffMem": {"IL7R", "LTB", "MAL", "AQP3", "ANXA1", "GIMAP7"},
}

MSC_MARKER_DB = {
    "Endothelial": {"PECAM1", "VWF", "CDH5", "CLDN5", "KDR", "FLT1"},
    "Fibro_iCAF": {"CXCL14", "CXCL12", "DCN", "LUM", "PDGFRA", "COL1A1"},
    "Fibro_myCAF": {"ACTA2", "TAGLN", "MYLK", "COL1A1", "COL3A1", "TPM2"},
    "Fibro_PVL": {"RGS5", "MCAM", "ACTA2", "TAGLN", "CSPG4", "MYH11"},
    "Fibro_Cycling": {"MKI67", "TOP2A", "CDK1", "BIRC5", "PCNA", "TYMS"},
}

MOUSE_B_MARKER_DB = {
    "Mature_B": {"Cd79a", "Cd79b", "Ms4a1", "Cd74", "H2-Aa", "H2-Ab1"},
    "Erythrocyte_like": {"Hba-a1", "Hba-a2", "Hbb-bs", "Hbb-bt", "Gypa", "Alas2"},
    "Mast_like": {"Cpa3", "Mcpt8", "Kit", "Ms4a2", "Gata2", "Tpsb2"},
    "pDC_Myeloid_like": {"Fcer1a", "Clec4c", "Bst2", "Irf8", "Tcf4", "Siglech"},
}

MARKER_DB_MAP = {
    "CD8": CD8_MARKER_DB,
    "CD4": CD4_MARKER_DB,
    "MSC": MSC_MARKER_DB,
    "MOUSE_B": MOUSE_B_MARKER_DB,
}

In [5]:
# ---------------------------------------------------------------------
# Official noise definitions
# ---------------------------------------------------------------------
COMPILED_NOISE_PATTERNS = {
    name: re.compile(pattern)
    for name, pattern in NOISE_PATTERNS.items()
}

NOISE_GENE_SET = set()
for _, genes in NOISE_LISTS.items():
    NOISE_GENE_SET.update(genes)


def is_regex_noise_gene(gene: str) -> bool:
    g = str(gene)
    if g in NOISE_GENE_SET:
        return True
    return any(p.search(g) for p in COMPILED_NOISE_PATTERNS.values())


def build_low_gini_gene_set(curator):
    """
    Use the package's official low-Gini housekeeping logic.
    """
    if curator.masker is None:
        raise ValueError("curator.masker is None; run curator.set_global_context(adata) first.")

    reason_map = curator.masker.detect_biological_noise(
        gini_threshold=None,
        gini_q=0.01,
        mean_floor=0.01,
        whitelist=[],
        rescue_mean_floor=0.05,
        low_gini_cap=0.15,
    )

    low_gini_genes = {
        g for g, reason in reason_map.items()
        if "Low_Gini" in str(reason)
    }
    return low_gini_genes


def build_gene_gini_map(curator):
    """
    gene -> global Gini coefficient
    """
    if curator.masker is None:
        raise ValueError("curator.masker is None; run curator.set_global_context(adata) first.")

    if curator.masker.gene_stats is None:
        curator.masker.calculate_gene_stats()

    gs = curator.masker.gene_stats.copy()
    return gs["gini"].to_dict()


def is_any_noise_gene(gene: str, low_gini_gene_set: set) -> bool:
    return is_regex_noise_gene(gene) or (str(gene) in low_gini_gene_set)


# ---------------------------------------------------------------------
# DE table builder
# ---------------------------------------------------------------------
def build_de_table(
    adata,
    cluster_name,
    cluster_col="meta.cluster",
    min_target_mean=0.02,
    min_delta_mean=0.02,
    min_logfc=0.2,
    min_target_pct=0.02,
    min_delta_pct=0.02,
):
    tmp = "__tmp_binary__"
    adata.obs[tmp] = "Rest"
    adata.obs.loc[adata.obs[cluster_col].astype(str) == str(cluster_name), tmp] = "Target"

    sc.tl.rank_genes_groups(
        adata,
        groupby=tmp,
        groups=["Target"],
        reference="Rest",
        method="wilcoxon",
        use_raw=False,
    )
    de_df_raw = sc.get.rank_genes_groups_df(adata, group="Target").copy()

    target_mask = (adata.obs[tmp] == "Target").values
    rest_mask = (adata.obs[tmp] == "Rest").values

    X = adata.X
    if issparse(X):
        X_target = X[target_mask, :]
        X_rest = X[rest_mask, :]
        target_mean = np.asarray(X_target.mean(axis=0)).ravel()
        rest_mean = np.asarray(X_rest.mean(axis=0)).ravel()
        target_pct = np.asarray((X_target > 0).mean(axis=0)).ravel()
        rest_pct = np.asarray((X_rest > 0).mean(axis=0)).ravel()
    else:
        X_target = X[target_mask, :]
        X_rest = X[rest_mask, :]
        target_mean = X_target.mean(axis=0)
        rest_mean = X_rest.mean(axis=0)
        target_pct = (X_target > 0).mean(axis=0)
        rest_pct = (X_rest > 0).mean(axis=0)

    expr_stats = pd.DataFrame({
        "names": adata.var_names,
        "target_mean": target_mean,
        "rest_mean": rest_mean,
        "target_pct": target_pct,
        "rest_pct": rest_pct,
    })

    de_df = de_df_raw.merge(expr_stats, on="names", how="left")
    de_df["delta_mean"] = de_df["target_mean"] - de_df["rest_mean"]
    de_df["delta_pct"] = de_df["target_pct"] - de_df["rest_pct"]

    eff_mask = (
        (de_df["target_mean"] >= min_target_mean) &
        (de_df["delta_mean"] >= min_delta_mean) &
        (de_df["target_pct"] >= min_target_pct) &
        (de_df["delta_pct"] >= min_delta_pct)
    )

    if "logfoldchanges" in de_df.columns:
        eff_mask &= (de_df["logfoldchanges"].fillna(0) >= min_logfc)

    de_df_filtered = de_df.loc[eff_mask].copy()

    adata.obs.drop(columns=[tmp], inplace=True, errors="ignore")
    return de_df_raw, de_df_filtered


# ---------------------------------------------------------------------
# Gene-list builders
# ---------------------------------------------------------------------
def get_standard_genes(de_df_raw, n_top=50):
    return de_df_raw["names"].head(n_top).tolist()


def get_filter_only_genes(de_df_raw, de_df_filtered, n_top=50):
    if de_df_filtered.shape[0] == 0:
        return de_df_raw["names"].head(n_top).tolist()
    return de_df_filtered["names"].head(n_top).tolist()


def get_regex_mask_genes(de_df_raw, de_df_filtered, n_top=50, oversample=300):
    base_df = de_df_filtered if de_df_filtered.shape[0] > 0 else de_df_raw
    genes = base_df["names"].head(oversample).tolist()
    clean = [g for g in genes if not is_regex_noise_gene(g)]
    return clean[:n_top]


def get_full_core_genes(curator, adata, cluster_name, cluster_col="meta.cluster", n_top=50):
    genes = curator.curate_features(
        adata,
        group_col=cluster_col,
        target_group=str(cluster_name),
        n_top=n_top,
        use_statistics=True,
    )
    return genes[:n_top]


# ---------------------------------------------------------------------
# Metrics
# ---------------------------------------------------------------------
def noise_fraction_any(genes, low_gini_gene_set):
    if len(genes) == 0:
        return np.nan
    return np.mean([is_any_noise_gene(g, low_gini_gene_set) for g in genes])


def low_gini_fraction(genes, low_gini_gene_set):
    if len(genes) == 0:
        return np.nan
    return np.mean([str(g) in low_gini_gene_set for g in genes])


def marker_recall_at_n(genes, gt_label, adata_var_names, marker_db):
    truth = marker_db.get(gt_label, set())
    var_set = set(map(str, adata_var_names))
    truth = [g for g in truth if g in var_set]
    if len(truth) == 0:
        return np.nan
    return sum(g in set(genes) for g in truth) / len(truth)


def mean_gene_gini(genes, gene_gini_map):
    vals = [gene_gini_map[g] for g in genes if g in gene_gini_map]
    if len(vals) == 0:
        return np.nan
    return float(np.mean(vals))

In [6]:
def prepare_adata_for_backend_free_metrics(adata):
    """
    Ensure:
    - GT-ready log1p .X
    - counts layer exists when possible
    - HVGs exist
    """
    if "counts" not in adata.layers:
        print("counts layer missing -> copying current X into adata.layers['counts']")
        adata.layers["counts"] = adata.X.copy()

    if "highly_variable" not in adata.var.columns:
        if "counts" in adata.layers:
            try:
                sc.pp.highly_variable_genes(
                    adata,
                    n_top_genes=min(2000, adata.n_vars),
                    flavor="seurat_v3",
                    layer="counts",
                    subset=False,
                )
                print("Computed HVGs from counts with seurat_v3")
            except Exception:
                sc.pp.highly_variable_genes(
                    adata,
                    n_top_genes=min(2000, adata.n_vars),
                    flavor="seurat",
                    subset=False,
                )
                print("Fallback: computed HVGs from current X with seurat")
        else:
            sc.pp.highly_variable_genes(
                adata,
                n_top_genes=min(2000, adata.n_vars),
                flavor="seurat",
                subset=False,
            )
            print("Computed HVGs from current X with seurat")

    return adata


def run_backend_free_dataset(cfg, n_top=50, topn_grid=(10, 20, 30, 50)):
    dataset = cfg["dataset"]
    path = cfg["path"]
    cluster_col = cfg["cluster_col"]
    gt_fn = cfg["gt_fn"]
    exclude_gt = cfg["exclude_gt"]
    marker_db = MARKER_DB_MAP[dataset]

    print(f"\n==================== {dataset} ====================")
    print("Loading:", path)

    adata = sc.read_h5ad(path)
    assert cluster_col in adata.obs.columns, f"{cluster_col} not found in adata.obs"

    adata = prepare_adata_for_backend_free_metrics(adata)
    adata.obs["Ground_Truth"] = adata.obs[cluster_col].astype(str).apply(gt_fn)

    curator = LLMscCurator(backend=DummyBackend())
    curator.set_global_context(adata)

    low_gini_gene_set = build_low_gini_gene_set(curator)
    gene_gini_map = build_gene_gini_map(curator)

    cluster_list = sorted(adata.obs[cluster_col].astype(str).unique())
    cluster_list = [c for c in cluster_list if gt_fn(c) not in exclude_gt]

    rows = []
    topn_rows = []
    gene_rows = []

    for i, cluster_name in enumerate(cluster_list, start=1):
        gt = gt_fn(cluster_name)
        print(f"[{i}/{len(cluster_list)}] {cluster_name} -> {gt}")

        de_df_raw, de_df_filtered = build_de_table(
            adata,
            cluster_name=cluster_name,
            cluster_col=cluster_col,
        )

        genes_standard = get_standard_genes(de_df_raw, n_top=n_top)
        genes_filter = get_filter_only_genes(de_df_raw, de_df_filtered, n_top=n_top)
        genes_regex = get_regex_mask_genes(de_df_raw, de_df_filtered, n_top=n_top, oversample=300)
        genes_core = get_full_core_genes(curator, adata, cluster_name, cluster_col=cluster_col, n_top=n_top)

        variant_to_genes = {
            "standard": genes_standard,
            "filter_only": genes_filter,
            "regex_mask": genes_regex,
            "full_core": genes_core,
        }

        # top-50 summary metrics
        for variant, genes in variant_to_genes.items():
            rows.append({
                "dataset": dataset,
                "cluster": cluster_name,
                "Ground_Truth": gt,
                "variant": variant,
                "noise_fraction": noise_fraction_any(genes, low_gini_gene_set),
                "low_gini_fraction": low_gini_fraction(genes, low_gini_gene_set),
                "marker_recall": marker_recall_at_n(genes, gt, adata.var_names, marker_db),
                "mean_gene_gini": mean_gene_gini(genes, gene_gini_map),
                "n_genes": len(genes),
            })

            for rank, g in enumerate(genes, start=1):
                gene_rows.append({
                    "dataset": dataset,
                    "cluster": cluster_name,
                    "Ground_Truth": gt,
                    "variant": variant,
                    "rank": rank,
                    "gene": g,
                    "is_regex_noise": is_regex_noise_gene(g),
                    "is_any_noise": is_any_noise_gene(g, low_gini_gene_set),
                    "is_low_gini": str(g) in low_gini_gene_set,
                    "is_canonical_marker": g in marker_db.get(gt, set()),
                })

        # top-N curves
        for n_eval in topn_grid:
            for variant, genes in variant_to_genes.items():
                genes_n = genes[:n_eval]
                topn_rows.append({
                    "dataset": dataset,
                    "cluster": cluster_name,
                    "Ground_Truth": gt,
                    "variant": variant,
                    "n_top_eval": n_eval,
                    "marker_recall_at_n": marker_recall_at_n(genes_n, gt, adata.var_names, marker_db),
                    "noise_fraction_at_n": noise_fraction_any(genes_n, low_gini_gene_set),
                    "low_gini_fraction_at_n": low_gini_fraction(genes_n, low_gini_gene_set),
                })

    df_metrics = pd.DataFrame(rows)
    df_topn = pd.DataFrame(topn_rows)
    df_genes = pd.DataFrame(gene_rows)

    df_metrics.to_csv(OUTPUT_DIR / f"{dataset.lower()}_backend_free_metrics.csv", index=False)
    df_topn.to_csv(OUTPUT_DIR / f"{dataset.lower()}_backend_free_topn_curve.csv", index=False)
    df_genes.to_csv(OUTPUT_DIR / f"{dataset.lower()}_backend_free_gene_lists.csv", index=False)

    return df_metrics, df_topn, df_genes

In [7]:
all_metrics = []
all_topn = []
all_genes = []

for cfg in DATASET_CONFIGS:
    df_metrics_i, df_topn_i, df_genes_i = run_backend_free_dataset(
        cfg,
        n_top=N_TOP,
        topn_grid=TOPN_GRID,
    )
    all_metrics.append(df_metrics_i)
    all_topn.append(df_topn_i)
    all_genes.append(df_genes_i)

df_metrics_all = pd.concat(all_metrics, ignore_index=True)
df_topn_all = pd.concat(all_topn, ignore_index=True)
df_genes_all = pd.concat(all_genes, ignore_index=True)

df_metrics_all.to_csv(OUTPUT_DIR / "FigS3_metrics.csv", index=False)
df_topn_all.to_csv(OUTPUT_DIR / "FigS3_topn.csv", index=False)
df_genes_all.to_csv(OUTPUT_DIR / "FigS3_gene_lists.csv", index=False)

print("\nSaved merged files:")
print(OUTPUT_DIR / "FigS3_metrics.csv")
print(OUTPUT_DIR / "FigS3_topn.csv")
print(OUTPUT_DIR / "FigS3_gene_lists.csv")

print("\nHead of merged metrics:")
display(df_metrics_all.head())


==================== CD8 ====================
Loading: /work/paper/gb_resubmission/input/cd8_benchmark_data.h5ad
Computed HVGs from counts with seurat_v3
[1/17] CD8.c01.Tn.MAL -> CD8_Naive
[2/17] CD8.c02.Tm.IL7R -> CD8_EffectorMemory
[3/17] CD8.c03.Tm.RPS12 -> CD8_EffectorMemory
[4/17] CD8.c04.Tm.CD52 -> CD8_EffectorMemory
[5/17] CD8.c05.Tem.CXCR5 -> CD8_EffectorMemory
[6/17] CD8.c06.Tem.GZMK -> CD8_EffectorMemory
[7/17] CD8.c07.Temra.CX3CR1 -> CD8_Effector
[8/17] CD8.c08.Tk.TYROBP -> CD8_Effector
[9/17] CD8.c09.Tk.KIR2DL4 -> CD8_Effector
[10/17] CD8.c10.Trm.ZNF683 -> CD8_EffectorMemory
[11/17] CD8.c11.Tex.PDCD1 -> CD8_Exhausted
[12/17] CD8.c12.Tex.CXCL13 -> CD8_Exhausted
[13/17] CD8.c13.Tex.myl12a -> CD8_Exhausted
[14/17] CD8.c14.Tex.TCF7 -> CD8_Exhausted
[15/17] CD8.c15.ISG.IFIT1 -> CD8_ISG
[16/17] CD8.c16.MAIT.SLC4A10 -> CD8_MAIT
[17/17] CD8.c17.Tm.NME1 -> CD8_EffectorMemory

==================== CD4 ====================
Loading: /work/paper/gb_resubmission/input/cd4_benchmark_data

,dataset,cluster,Ground_Truth,variant,noise_fraction,low_gini_fraction,marker_recall,mean_gene_gini,n_genes
0,CD8,CD8.c01.Tn.MAL,CD8_Naive,standard,0.88,0.02,0.500,0.172520,50
1,CD8,CD8.c01.Tn.MAL,CD8_Naive,filter_only,0.50,0.02,0.875,0.410391,50
2,CD8,CD8.c01.Tn.MAL,CD8_Naive,regex_mask,0.04,0.04,0.875,0.592995,50
3,CD8,CD8.c01.Tn.MAL,CD8_Naive,full_core,0.00,0.00,1.000,0.737540,50
4,CD8,CD8.c02.Tm.IL7R,CD8_EffectorMemory,standard,0.24,0.08,0.500,0.402780,50


In [8]:
summary_all = (
    df_metrics_all
    .groupby(["dataset", "variant"], as_index=False)[
        ["noise_fraction", "low_gini_fraction", "marker_recall", "mean_gene_gini"]
    ]
    .mean()
)

summary_all["noise_pct"] = summary_all["noise_fraction"] * 100
summary_all["low_gini_pct"] = summary_all["low_gini_fraction"] * 100
summary_all["marker_recall_pct"] = summary_all["marker_recall"] * 100

topn_summary_all = (
    df_topn_all
    .groupby(["dataset", "variant", "n_top_eval"], as_index=False)[
        ["marker_recall_at_n", "noise_fraction_at_n", "low_gini_fraction_at_n"]
    ]
    .mean()
)

topn_summary_all["marker_recall_pct"] = topn_summary_all["marker_recall_at_n"] * 100
topn_summary_all["noise_pct"] = topn_summary_all["noise_fraction_at_n"] * 100
topn_summary_all["low_gini_pct"] = topn_summary_all["low_gini_fraction_at_n"] * 100

display(summary_all)
display(topn_summary_all.head())

,dataset,variant,noise_fraction,low_gini_fraction,marker_recall,mean_gene_gini,noise_pct,low_gini_pct,marker_recall_pct
0,CD4,filter_only,0.085455,0.004545,0.522727,0.574426,8.545455,0.454545,52.272727
1,CD4,full_core,0.000000,0.000000,0.515152,0.714163,0.000000,0.000000,51.515152
2,CD4,regex_mask,0.004545,0.004545,0.528409,0.598100,0.454545,0.454545,52.840909
3,CD4,standard,0.307273,0.052727,0.441288,0.438955,30.727273,5.272727,44.128788
4,CD8,filter_only,0.092101,0.007059,0.529412,0.586759,9.210084,0.705882,52.941176
5,CD8,full_core,0.001176,0.001176,0.536765,0.684021,0.117647,0.117647,53.676471
6,CD8,regex_mask,0.008235,0.008235,0.536765,0.601310,0.823529,0.823529,53.676471
7,CD8,standard,0.234118,0.042353,0.500000,0.482145,23.411765,4.235294,50.000000
8,MOUSE_B,filter_only,0.215000,0.005000,0.866667,0.547758,21.500000,0.500000,86.666667
9,MOUSE_B,full_core,0.005000,0.000000,0.700000,0.755417,0.500000,0.000000,70.000000


,dataset,variant,n_top_eval,marker_recall_at_n,noise_fraction_at_n,low_gini_fraction_at_n,marker_recall_pct,noise_pct,low_gini_pct
0,CD4,filter_only,10,0.310606,0.100000,0.009091,31.060606,10.000000,0.909091
1,CD4,filter_only,20,0.412879,0.097727,0.009091,41.287879,9.772727,0.909091
2,CD4,filter_only,30,0.464015,0.090909,0.006061,46.401515,9.090909,0.606061
3,CD4,filter_only,50,0.522727,0.085455,0.004545,52.272727,8.545455,0.454545
4,CD4,full_core,10,0.339015,0.000000,0.000000,33.901515,0.000000,0.000000


In [9]:
plt.rcParams["font.family"] = "sans-serif"
plt.rcParams["font.sans-serif"] = ["Arial", "DejaVu Sans"]

order = ["standard", "filter_only", "regex_mask", "full_core"]
xpos = np.arange(len(order))

color_map = {
    "standard": "#BDBDBD",
    "filter_only": "#F2B134",
    "regex_mask": "#42A5F5",
    "full_core": "#D32F2F",
}

dataset_order = ["CD8", "CD4", "MSC", "MOUSE_B"]
dataset_display = {
    "CD8": "CD8",
    "CD4": "CD4",
    "MSC": "MSC",
    "MOUSE_B": "Mouse B",
}

def slope_panel(ax, metric_col, ylabel, xlabel):
    for ds in dataset_order:
        sub = summary_all[summary_all["dataset"] == ds].set_index("variant").loc[order].reset_index()
        y = sub[metric_col].values

        # thin black slope per dataset
        ax.plot(
            xpos, y,
            color="black",
            linewidth=1.0,
            alpha=0.35,
            zorder=1,
        )

        # colored points by variant
        for i, v in enumerate(order):
            ax.scatter(
                xpos[i], y[i],
                s=48,
                color=color_map[v],
                edgecolor="black",
                linewidth=0.7,
                zorder=2,
            )

        # annotate dataset at full_core end
        ax.text(
            xpos[0] - 0.08, y[0],
            dataset_display[ds],
            fontsize=8,
            va="center",
            ha="right",
            color="black",
            alpha=0.8,
        )

    ax.set_xticks(xpos)
    ax.set_xticklabels(order, rotation=20, ha="right")
    ax.set_ylabel(ylabel)
    ax.set_xlabel(xlabel, fontweight="bold")
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.set_xlim(-0.7, len(order) - 0.7)

fig, axes = plt.subplots(1, 3, figsize=(14, 4.5), dpi=250)

slope_panel(
    axes[0],
    metric_col="noise_pct",
    ylabel="Noise fraction (%)",
    xlabel="Biological-noise fraction across datasets",
)

slope_panel(
    axes[1],
    metric_col="low_gini_pct",
    ylabel="Low-Gini fraction (%)",
    xlabel="Low-Gini housekeeping fraction across datasets",
)

slope_panel(
    axes[2],
    metric_col="marker_recall_pct",
    ylabel="Canonical marker recall (%)",
    xlabel="Canonical marker recall across datasets",
)


plt.tight_layout()
plt.savefig(OUTPUT_DIR / "FigS3b.png", dpi=300, bbox_inches="tight")
plt.savefig(OUTPUT_DIR / "FigS3b.pdf", bbox_inches="tight")
plt.show()

In [10]:
plt.rcParams["font.family"] = "sans-serif"
plt.rcParams["font.sans-serif"] = ["Arial", "DejaVu Sans"]

fig, axes = plt.subplots(1, 4, figsize=(14, 4.8), dpi=250, sharey=True)

for ax, ds in zip(axes, dataset_order):
    sub_ds = topn_summary_all[topn_summary_all["dataset"] == ds].copy()

    for variant in order:
        sub = sub_ds[sub_ds["variant"] == variant].sort_values("n_top_eval")
        ax.plot(
            sub["n_top_eval"],
            sub["marker_recall_pct"],
            marker="o",
            linewidth=2.0,
            markersize=5,
            color=color_map[variant],
            label=variant,
        )

    ax.set_title(dataset_display[ds], fontweight="bold")
    ax.set_xlabel("Top N genes")
    ax.set_xticks(TOPN_GRID)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

axes[3].set_ylabel("Canonical marker recall (%)")
axes[3].legend(frameon=False, fontsize=8, loc="lower right")

fig.suptitle("d  Canonical marker recall across input-list length", y=1.02, fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "FigS3c.png", dpi=300, bbox_inches="tight")
plt.savefig(OUTPUT_DIR / "FigS3c.pdf", bbox_inches="tight")
plt.show()

In [11]:
plt.rcParams["font.family"] = "sans-serif"
plt.rcParams["font.sans-serif"] = ["Arial", "DejaVu Sans"]

fig, axes = plt.subplots(1, 4, figsize=(14, 4.8), dpi=250, sharey=True)

for ax, ds in zip(axes, dataset_order):
    sub_ds = topn_summary_all[topn_summary_all["dataset"] == ds].copy()

    for variant in order:
        sub = sub_ds[sub_ds["variant"] == variant].sort_values("n_top_eval")
        ax.plot(
            sub["n_top_eval"],
            sub["noise_pct"],
            marker="o",
            linewidth=2.0,
            markersize=5,
            color=color_map[variant],
            label=variant,
        )

    ax.set_title(dataset_display[ds], fontweight="bold")
    ax.set_xlabel("Top N genes")
    ax.set_xticks(TOPN_GRID)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

axes[3].set_ylabel("Noise fraction (%)")
axes[3].legend(frameon=False, fontsize=8, loc="upper right")

fig.suptitle("Optional: biological-noise fraction across input-list length", y=1.02, fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "FigS3a.png", dpi=300, bbox_inches="tight")
plt.savefig(OUTPUT_DIR / "FigS3a.pdf", bbox_inches="tight")
plt.show()